# EXP-28: Scaffold Energy Contribution
**ΔG_scaffold = ΔG_full(BPTI) − ΔG_peptide(loop 13–20)**

- **Expected:** −7.9 ± 3.0 kcal/mol
- **PASS:** [−10.9, −4.9] | **MARGINAL:** [−13.9, −1.9] | **FAIL:** outside
- **Runs peptide–trypsin SMD + US on GPU, loads ΔG_full from EXP-04**

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-28'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['prep','simulation','analysis','figures']:
    (WORK_DIR/d).mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO)

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')

def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mdtraj as md
from src.config import (SystemConfig, MinimizationConfig, EquilibrationConfig,
                        ProductionConfig, SMDConfig, UmbrellaConfig, WHAMConfig, KCAL_TO_KJ)
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.production import run_production
from src.simulate.smd import run_smd_campaign
from src.simulate.umbrella import run_umbrella_campaign
from src.simulate.platform import select_platform
from src.analyze.jarzynski import jarzynski_free_energy
from src.analyze.wham import solve_wham
from openmm.app import PME, ForceField, Simulation, HBonds, PDBFile, Modeller
from openmm import LangevinMiddleIntegrator, XmlSerializer
print('Imports OK')

In [ ]:
TEMPERATURE_K = 310.0
system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(nvt_duration_ps=500.0, npt_duration_ps=1000.0, temperature_k=TEMPERATURE_K)
prod_config = ProductionConfig(duration_ns=10.0, temperature_k=TEMPERATURE_K)
smd_config = SMDConfig(n_replicates=30, pull_distance_nm=3.0,  # optimized from 50
                       spring_constant_kj_mol_nm2=1000.0)
us_config = UmbrellaConfig(xi_min_nm=1.5, xi_max_nm=6.5, window_spacing_nm=0.1,
                           spring_constant_kj_mol_nm2=1000.0, per_window_duration_ns=2.0)
wham_config = WHAMConfig()

# Load EXP-04 delta_G_full
EXP04_DIR = Path('/content/drive/MyDrive/v3_gpu_results/EXP-04')
dg_full = -18.0  # expected fallback
exp04_results = EXP04_DIR / 'results.json'
if exp04_results.exists():
    with open(exp04_results) as f:
        r04 = json.load(f)
    dg_full = r04.get('delta_g_us_kcal', r04.get('delta_g_kcal_mol', dg_full))
    print(f'Loaded EXP-04 \u0394G_full = {dg_full:.2f} kcal/mol')
else:
    print(f'Using expected \u0394G_full = {dg_full} kcal/mol')

In [ ]:
# Build peptide-trypsin system
# Strategy: fetch 2PTC, extract binding loop (BPTI resids 13-20) + trypsin chain
PREP_DIR = WORK_DIR / 'prep'
raw_pdb = fetch_pdb('2PTC', output_dir=PREP_DIR)
cleaned = clean_structure(raw_pdb, chains_to_keep=['E', 'I'], remove_heteroatoms=True, remove_waters=True)

# Extract loop peptide using mdtraj
full_struct = md.load(str(cleaned))
# Identify BPTI chain (chain I) residues 13-20, plus trypsin chain (chain E)
bpti_loop_atoms = full_struct.topology.select('chainid 1 and resid 12 to 19')  # 0-indexed
trypsin_atoms = full_struct.topology.select('chainid 0')
keep_atoms = np.concatenate([trypsin_atoms, bpti_loop_atoms])
keep_atoms.sort()

peptide_struct = full_struct.atom_slice(keep_atoms)
peptide_pdb = str(PREP_DIR / 'peptide_trypsin.pdb')
peptide_struct.save(peptide_pdb)
print(f'Peptide-trypsin complex: {peptide_struct.n_atoms} atoms')

# Identify pull groups in the peptide-trypsin system
trypsin_idx = []
peptide_idx = []
for chain in peptide_struct.topology.chains:
    atoms = [a.index for a in chain.atoms]
    if chain.index == 0:
        trypsin_idx = atoms
    else:
        peptide_idx = atoms
print(f'Trypsin: {len(trypsin_idx)} atoms, Peptide: {len(peptide_idx)} atoms')

In [ ]:
# Build system from peptide-trypsin PDB
topology_pept, system_pept, modeller_pept = build_topology(
    Path(peptide_pdb), system_config)
modeller_pept, n_water, _, _ = solvate_system(modeller_pept, system_config)
print(f'Solvated: {n_water} waters')

ff = ForceField(system_config.force_field, system_config.water_model)
system_pept = ff.createSystem(modeller_pept.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
integrator = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
simulation = Simulation(modeller_pept.topology, system_pept, integrator, select_platform('CUDA'))
simulation.context.setPositions(modeller_pept.positions)

topo_pdb = PREP_DIR / 'peptide_trypsin_solvated.pdb'
with open(topo_pdb, 'w') as f:
    PDBFile.writeFile(modeller_pept.topology, modeller_pept.positions, f)

min_result = minimize_energy(simulation, min_config)
print(f'Minimized: {min_result["final_energy_kj_mol"]:.0f} kJ/mol')

In [ ]:
SIM_DIR = WORK_DIR / 'simulation'
nvt_result = run_nvt(simulation, eq_config, SIM_DIR/'nvt')
npt_result = run_npt(simulation, eq_config, SIM_DIR/'npt')
print(f'NPT done: T={npt_result["avg_temperature_k"]:.1f} K')

# Save system XML and state for SMD/US
system_xml_path = SIM_DIR / 'system.xml'
with open(system_xml_path, 'w') as f:
    f.write(XmlSerializer.serialize(system_pept))

npt_state = simulation.context.getState(getPositions=True, getVelocities=True)
state_xml_path = SIM_DIR / 'npt_final_state.xml'
with open(state_xml_path, 'w') as f:
    f.write(XmlSerializer.serialize(npt_state))

# Rebuild pull groups based on solvated topology
solvated_top = md.load(str(topo_pdb)).topology
chain_list = list(solvated_top.chains)
pull_group_1 = [a.index for a in chain_list[0].atoms]  # trypsin
pull_group_2 = [a.index for a in chain_list[1].atoms]  # peptide
print(f'Pull groups: {len(pull_group_1)} + {len(pull_group_2)} atoms')

In [ ]:
# SMD campaign
if ckpt.is_done('smd'):
    print('⏭ SMD already completed, skipping')
    dg_jarz_kcal = ckpt.get_data('smd').get('dg_jarz_kcal', 0)
else:
    smd_results = run_smd_campaign(
        equilibrated_state_path=state_xml_path, system_xml_path=system_xml_path,
        config=smd_config, pull_group_1=pull_group_1, pull_group_2=pull_group_2,
        output_dir=SIM_DIR/'smd', topology_pdb_path=topo_pdb, platform_name='CUDA')

    work_vals = np.array([r['total_work_kj_mol'] for r in smd_results])
    jarz = jarzynski_free_energy(work_vals, TEMPERATURE_K)
    dg_jarz_kj = jarz['delta_g_kj_mol']
    dg_jarz_kcal = dg_jarz_kj / KCAL_TO_KJ
    print(f'Jarzynski ΔG (peptide): {dg_jarz_kcal:.2f} kcal/mol')
    ckpt.mark_done('smd', {'dg_jarz_kcal': dg_jarz_kcal})

In [ ]:
# Umbrella sampling campaign
if ckpt.is_done('umbrella'):
    print('⏭ Umbrella sampling already completed, skipping')
    dg_us_kcal = ckpt.get_data('umbrella').get('dg_us_kcal', 0)
else:
    us_results = run_umbrella_campaign(
        equilibrated_state_path=state_xml_path, system_xml_path=system_xml_path,
        config=us_config, pull_group_1=pull_group_1, pull_group_2=pull_group_2,
        output_dir=SIM_DIR/'umbrella', topology_pdb_path=topo_pdb, platform_name='CUDA')

    xi_ts_list = [np.array(r['xi_timeseries']) for r in us_results]
    centers = np.array([r['window_center_nm'] for r in us_results])
    springs = np.array([r['spring_constant_kj_mol_nm2'] for r in us_results])

    pmf = solve_wham(xi_ts_list, centers, springs, TEMPERATURE_K, wham_config)
    pmf_kj = np.array(pmf['pmf_kj_mol'])
    pmf_xi = np.array(pmf['bin_centers_nm'])
    pmf_kcal = pmf_kj / KCAL_TO_KJ
    dg_us_kcal = float(pmf_kcal[-1] - pmf_kcal[0])
    print(f'US/WHAM ΔG (peptide): {dg_us_kcal:.2f} kcal/mol')
    ckpt.mark_done('umbrella', {'dg_us_kcal': dg_us_kcal, 'n_windows': len(us_results)})

In [ ]:
# Scaffold energy = full - peptide
dg_peptide = dg_us_kcal  # use US/WHAM value as primary
dg_scaffold = dg_full - dg_peptide

print(f'\u0394G_full (EXP-04):     {dg_full:.2f} kcal/mol')
print(f'\u0394G_peptide (US):     {dg_peptide:.2f} kcal/mol')
print(f'\u0394G_scaffold:         {dg_scaffold:.2f} kcal/mol')

if -10.9 <= dg_scaffold <= -4.9:
    verdict = 'PASS'
elif -13.9 <= dg_scaffold <= -1.9:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'\nVERDICT: {verdict} (expected [-10.9, -4.9])')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(pmf_xi, pmf_kcal, 'b-', lw=2)
ax.set_xlabel('\u03be (nm)'); ax.set_ylabel('PMF (kcal/mol)')
ax.set_title('Peptide\u2013Trypsin PMF')

ax = axes[1]
labels = ['\u0394G_full', '\u0394G_peptide', '\u0394G_scaffold']
vals = [dg_full, dg_peptide, dg_scaffold]
colors = ['steelblue', 'coral', '#2ecc71']
ax.bar(labels, vals, color=colors, edgecolor='navy')
ax.axhspan(-10.9, -4.9, alpha=0.1, color='green', label='PASS range')
ax.set_ylabel('kcal/mol'); ax.set_title(f'Energy Decomposition \u2014 {verdict}')
ax.legend()

fig.suptitle('EXP-28: Scaffold Energy', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
sync_stop.set()  # Stop periodic sync before final copy

results = {
    'experiment_id': EXP_ID, 'dg_full_kcal': round(dg_full, 3),
    'dg_peptide_kcal': round(dg_peptide, 3), 'dg_scaffold_kcal': round(dg_scaffold, 3),
    'dg_jarz_peptide_kcal': round(dg_jarz_kcal, 3), 'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2)

# Save PMF data
np.savez(WORK_DIR/'analysis'/'pmf_data.npz', xi=pmf_xi, pmf_kcal=pmf_kcal)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

ckpt.mark_done('complete')

# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))